# Hybrid Admission Pre-Screening System
## Machine Learning Model Training — Bingham University

This notebook demonstrates the complete machine learning pipeline:
1. Dataset loading from CSV
2. Data preprocessing and label encoding
3. Rule-based screening logic demonstration
4. Comparative model evaluation (Decision Tree, Logistic Regression, Random Forest)
5. Random Forest training (selected best model)
6. Classification report and confusion matrix
7. Model serialization using Joblib

## Cell 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import joblib
import os

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
print('All libraries imported successfully.')
print(f'Pandas version  : {pd.__version__}')
print(f'NumPy version   : {np.__version__}')
print(f'Joblib version  : {joblib.__version__}')

All libraries imported successfully.
Pandas version  : 3.0.3
NumPy version   : 2.3.5
Joblib version  : 1.5.3


## Cell 2: Load Dataset from CSV

In [1]:
# Load the simulated admission dataset
CSV_PATH = os.path.join('..', 'data', 'admission_dataset.csv')
data = pd.read_csv(CSV_PATH)

print('Dataset loaded successfully!')
print(f'Shape: {data.shape}  ({data.shape[0]} records, {data.shape[1]} features)')
print()
data.head(10)

NameError: name 'os' is not defined

In [4]:
# Dataset overview
print('=== Dataset Info ===')
data.info()
print()
print('=== Statistical Summary ===')
data[['utme_score', 'olevel_avg_score', 'departmental_cutoff']].describe()

=== Dataset Info ===


NameError: name 'data' is not defined

In [5]:
# Outcome class distribution
print('=== Admission Outcome Distribution ===')
print(data['outcome'].value_counts())
print()

fig, ax = plt.subplots(figsize=(8, 4))
data['outcome'].value_counts().plot(
    kind='bar', ax=ax,
    color=['#3b82f6', '#ef4444', '#10b981', '#f59e0b']
)
ax.set_title('Dataset — Admission Outcome Distribution')
ax.set_xlabel('Outcome')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

=== Admission Outcome Distribution ===


NameError: name 'data' is not defined

## Cell 3: Data Preprocessing

In [6]:
print('=== Step 1: Handle Missing Values ===')
print(f'Missing values before cleaning:\n{data.isnull().sum()}')

drop_cols = [
    'applicant_name', 'jamb_reg_number',
    'olevel_math', 'olevel_english',
    'olevel_subject_1', 'olevel_subject_2', 'olevel_subject_3',
    'olevel_grade_1', 'olevel_grade_2', 'olevel_grade_3'
]
data_clean = data.drop(columns=[c for c in drop_cols if c in data.columns])
data_clean = data_clean.dropna()

print(f'\nShape after cleaning: {data_clean.shape}')
print(f'Missing values after cleaning: {data_clean.isnull().sum().sum()}')

=== Step 1: Handle Missing Values ===


NameError: name 'data' is not defined

In [ ]:
print('=== Step 2: Label Encoding ===')

# Encode course names
le_course = LabelEncoder()
data_clean = data_clean.copy()
data_clean['course_applied'] = le_course.fit_transform(data_clean['course_applied'])

# Encode outcome labels
le_outcome = LabelEncoder()
data_clean['outcome_encoded'] = le_outcome.fit_transform(data_clean['outcome'])

print('Outcome label encoding:')
for cls, idx in zip(le_outcome.classes_, le_outcome.transform(le_outcome.classes_)):
    print(f'  {cls:25s} -> {idx}')

print()
data_clean[['utme_score','olevel_avg_score','course_applied',
            'departmental_cutoff','outcome_encoded']].head(10)

In [ ]:
print('=== Step 3: Feature Selection and Data Splitting ===')

FEATURES = ['utme_score', 'olevel_avg_score', 'course_applied', 'departmental_cutoff']
X = data_clean[FEATURES]
y = data_clean['outcome_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Selected features : {FEATURES}')
print(f'Training set size : {len(X_train)} samples (80%)')
print(f'Testing  set size : {len(X_test)}  samples (20%)')
print()
X_train.describe()

## Cell 4: Rule-Based Screening Logic Demonstration

In [ ]:
# Demonstrate rule-based screening on sample applicants
GRADE_WEIGHTS = {
    'A1': 5, 'B2': 4, 'B3': 4, 'C4': 3, 'C5': 3, 'C6': 3,
    'D7': 2, 'E8': 1, 'F9': 0
}

DEPT_CUTOFFS = {
    'Computer Science': 200,
    'Medicine and Surgery': 280,
    'Law': 220,
    'Accounting': 200,
    'Nursing Science': 200,
    'Mass Communication': 180,
}

def rule_based_screen(applicant):
    """Applies institutional and departmental admission rules."""
    utme        = applicant['utme_score']
    course      = applicant['course_applied']
    grades      = applicant['o_level_grades']   # list of 5 grade strings
    
    weights     = [GRADE_WEIGHTS.get(g.upper(), 0) for g in grades]
    credit_pass = sum(1 for w in weights if w >= 3)
    has_math    = weights[0] >= 3
    has_english = weights[1] >= 3
    dept_cutoff = DEPT_CUTOFFS.get(course, 180)
    
    # --- Institutional Requirements ---
    if utme < 140:
        return {'status': 'REJECTED', 'reason': 'UTME score below minimum institutional cut-off (140).',
                'passed_institutional': False, 'passed_departmental': False}
    if not has_math:
        return {'status': 'REJECTED', 'reason': 'Credit pass in Mathematics is required.',
                'passed_institutional': False, 'passed_departmental': False}
    if not has_english:
        return {'status': 'REJECTED', 'reason': 'Credit pass in English Language is required.',
                'passed_institutional': False, 'passed_departmental': False}
    if credit_pass < 5:
        return {'status': 'REJECTED', 'reason': f'Minimum 5 credit passes required. Got {credit_pass}.',
                'passed_institutional': False, 'passed_departmental': False}
    
    # --- Departmental Requirements ---
    if utme >= dept_cutoff:
        return {'status': 'QUALIFIED',
                'reason': f'Applicant meets all requirements for {course}.',
                'passed_institutional': True, 'passed_departmental': True}
    elif utme >= dept_cutoff - 20:
        return {'status': 'BORDERLINE',
                'reason': f'Score is within borderline range for {course}. Subject to ML review.',
                'passed_institutional': True, 'passed_departmental': False}
    else:
        return {'status': 'ALTERNATIVE_COURSE',
                'reason': f'Does not meet {course} cut-off. Recommended: Mass Communication.',
                'passed_institutional': True, 'passed_departmental': False}

# --- Test applicants ---
test_applicants = [
    {'name': 'Emmanuel Okafor',  'utme_score': 265, 'course_applied': 'Computer Science',
     'o_level_grades': ['A1', 'B2', 'B3', 'C4', 'C5']},
    {'name': 'Fatima Bello',     'utme_score': 195, 'course_applied': 'Computer Science',
     'o_level_grades': ['B2', 'B3', 'A1', 'C4', 'B2']},
    {'name': 'Ngozi Nwosu',      'utme_score': 310, 'course_applied': 'Medicine and Surgery',
     'o_level_grades': ['A1', 'A1', 'A1', 'B2', 'B3']},
    {'name': 'Daniel Ibrahim',   'utme_score': 155, 'course_applied': 'Law',
     'o_level_grades': ['D7', 'C4', 'C5', 'E8', 'D7']},
    {'name': 'Blessing Adeleke', 'utme_score': 172, 'course_applied': 'Accounting',
     'o_level_grades': ['C4', 'B3', 'C5', 'C6', 'B2']},
]

print('=== Rule-Based Admission Screening Results ===')
print(f'{"Applicant Name":<25} {"UTME":>6}  {"Course":<25} {"Status":<20}  Reason')
print('-' * 110)
for a in test_applicants:
    result = rule_based_screen(a)
    print(f'{a["name"]:<25} {a["utme_score"]:>6}  {a["course_applied"]:<25} {result["status"]:<20}  {result["reason"]}')

## Cell 5: Comparative Model Evaluation

In [ ]:
print('=== Comparative Model Evaluation ===')

classifiers = {
    'Decision Tree':       DecisionTreeClassifier(max_depth=6, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
}

eval_results = {}
for name, model in classifiers.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    eval_results[name] = acc
    print(f'  {name:25s}  Accuracy: {acc:.4f}  ({acc*100:.2f}%)')

print()
best_model_name = max(eval_results, key=eval_results.get)
print(f'Selected Model: {best_model_name} (Accuracy: {eval_results[best_model_name]*100:.2f}%)')

# Bar chart comparison
fig, ax = plt.subplots(figsize=(8, 4))
bar_colors = ['#94a3b8', '#94a3b8', '#3b82f6']
bars = ax.bar(eval_results.keys(), [v * 100 for v in eval_results.values()], color=bar_colors)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Comparative Model Evaluation — Accuracy')
ax.set_ylim(0, 110)
for bar, acc in zip(bars, eval_results.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{acc*100:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## Cell 6: Random Forest Training and Prediction

In [ ]:
print('=== Random Forest Model Training ===')

# Initialize Random Forest
clf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
print(f'Model initialized: {clf}')

# Train the model
clf.fit(X_train, y_train)
print(f'\nModel trained on {len(X_train)} samples.')

# Predict on test set
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)

# Accuracy
acc = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {acc:.4f} ({acc*100:.2f}%)')

# Show first 10 predictions with confidence
print()
print('=== Sample Predictions (first 10 test records) ===')
print(f'{"Actual":<25} {"Predicted":<25} {"Confidence":>12}')
print('-' * 65)
for actual, pred, proba in zip(y_test[:10], y_pred[:10], y_proba[:10]):
    actual_label = le_outcome.inverse_transform([actual])[0]
    pred_label   = le_outcome.inverse_transform([pred])[0]
    confidence   = proba.max() * 100
    match        = '<<' if actual != pred else ''
    print(f'{actual_label:<25} {pred_label:<25} {confidence:>11.1f}%  {match}')

In [ ]:
# Prediction outcomes distribution
class_names = le_outcome.classes_
pred_labels = le_outcome.inverse_transform(y_pred)
outcome_series = pd.Series(pred_labels).value_counts()

fig, ax = plt.subplots(figsize=(8, 5))
outcome_series.plot(kind='bar', ax=ax,
                    color=['#3b82f6', '#ef4444', '#10b981', '#f59e0b'])
ax.set_title('Predicted Admission Outcomes Distribution')
ax.set_xlabel('Admission Status')
ax.set_ylabel('Number of Applicants')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

## Cell 7: Classification Report and Confusion Matrix

In [ ]:
print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
print('=== Confusion Matrix ===')

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues', colorbar=True)
ax.set_title('Confusion Matrix — Random Forest Classifier')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

print('\nRaw Confusion Matrix:')
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
print(cm_df)

## Cell 8: Feature Importance

In [ ]:
importances = pd.Series(clf.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 4))
importances.plot(kind='barh', ax=ax, color='#3b82f6')
ax.set_title('Random Forest — Feature Importance')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Feature Importances:')
for feat, imp in sorted(zip(FEATURES, clf.feature_importances_), key=lambda x: -x[1]):
    print(f'  {feat:<25} {imp:.4f}')

## Cell 9: Serialise Model with Joblib

In [ ]:
os.makedirs(os.path.join('..', 'models'), exist_ok=True)
model_path = os.path.join('..', 'models', 'random_forest_model.joblib')

joblib.dump({
    'model':           clf,
    'label_encoder':   le_outcome,
    'feature_encoder': le_course,
    'features':        FEATURES,
    'class_names':     list(class_names)
}, model_path)

print(f'Model successfully serialised with Joblib.')
print(f'Saved to: {os.path.abspath(model_path)}')

# Reload and verify
loaded = joblib.load(model_path)
test_prediction = loaded['model'].predict(X_test[:1])
print(f'\nVerification — prediction from reloaded model: {le_outcome.inverse_transform(test_prediction)[0]}')
print('Model reload successful. Training pipeline complete.')